# This notebook uses a GoogleNet model

## Imports

In [17]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset, random_split
from torch.optim.lr_scheduler import ReduceLROnPlateau



from torchvision import transforms, models

from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
import copy

from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix,classification_report
import numpy as np
from sklearn.model_selection import StratifiedKFold


from google.oauth2 import service_account
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

import pandas as pd


### Working directory setup

In [18]:
# Mappa elérési útja
results_dir = '/kaggle/working/results'

# Ellenőrizd, hogy a mappa létezik-e, és hozd létre, ha nem
if not os.path.exists(results_dir):
    os.makedirs(results_dir)
    print(f"Directory '{results_dir}' created.")

## Dataset class definition, inherited from PyTorch dataset class
A megadott Python osztály, ```MaskedImageDataset``` a PyTorch beépített Dataset osztályából származik, és képes képadatok és hozzájuk tartozó maszkok kezelésére. Az osztály ``__getitem__`` felülírt függvényének  kimenete az adathalmazban levő bináris maszkok felhasználásával adja vissza a maszkolt képeket. 

In [19]:
class MaskedImageDataset(Dataset):
    def __init__(self, data_dir, categories, transform=None, sample_size=None):
        self.image_paths = []
        self.mask_paths = []
        self.labels = []
        self.class_names = []
        self.transform = transform
        self.label_map = {cat: i for i, cat in enumerate(categories)}
        self.inv_label_map = {i: cat for cat, i in self.label_map.items()}  # Szám → név átalakítás

        for category in categories:
            images_path = os.path.join(data_dir, category, "images")
            masks_path = os.path.join(data_dir, category, "masks")
            
            if not os.path.isdir(images_path) or not os.path.isdir(masks_path):
                continue
            
            files = os.listdir(images_path)
            if sample_size:  # If sample_size is given, limit the number of files
                files = files[:sample_size]  
                
            for file in tqdm(files, desc=f"Loading {category} images"):
                img_path = os.path.join(images_path, file)
                mask_path = os.path.join(masks_path, file)
                
                if os.path.exists(mask_path):
                    self.image_paths.append(img_path)
                    self.mask_paths.append(mask_path)
                    self.labels.append(self.label_map[category])
                    self.class_names.append(category)

    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        mask_path = self.mask_paths[idx]
        label = self.labels[idx]
    
        # Load images and masks
        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")  # grayscale
    
        # Resize both image and mask
        fixed_size = (256, 256)
        image = image.resize(fixed_size, Image.BILINEAR)
        mask = mask.resize(fixed_size, Image.NEAREST)
    
        # Apply transforms
        if self.transform:
            image = self.transform(image)  # apply full transform to image
        mask = transforms.ToTensor()(mask)  # only convert mask to tensor
    
        # Convert mask to binary (0 or 1)
        mask = (mask > 0.5).float()
    
        # Ensure mask has 3 channels like the image
        mask = mask.expand(3, -1, -1)  # shape [3, 256, 256]
    
        # Ensure shape match
        assert image.shape == mask.shape, f"Shape mismatch: {image.shape} vs {mask.shape}"
    
        # Apply mask to image
        masked_image = image * mask
    
        return masked_image, label


### A MaskedImageDataset osztály használata

In [20]:
transform = transforms.Compose([
    transforms.Resize((256, 256)),  # ResNet bemeneti mérete
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

dataset = MaskedImageDataset(data_dir='/kaggle/input/covid19-radiography-database/COVID-19_Radiography_Dataset',
                             categories=["COVID", "Normal", "Viral Pneumonia", "Lung_Opacity"],
                             transform=transform,
                             sample_size=None)

num_classes = len(dataset.label_map)
print(num_classes)

Loading Lung_Opacity images: 100%|██████████| 6012/6012 [00:12<00:00, 469.13it/s]

4


### Adatok felosztása tanító és tesztelő halmazokra

In [21]:
# Define split ratios
train_ratio = 0.80
test_ratio = 0.20

total_size = len(dataset)

train_size = int(train_ratio * total_size)
test_size = int(test_ratio * total_size)

# train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

def count_per_category(dataset):
    category_counts = {cat: 0 for cat in dataset.dataset.label_map.keys()}
    
    # Wrap the loop with tqdm to show progress
    for _, label in tqdm(dataset, desc="Counting categories"):
        category_name = dataset.dataset.inv_label_map[label]
        category_counts[category_name] += 1
    
    return category_counts

# # Example usage
# print("Training Set Distribution:", count_per_category(train_dataset))
# print("Test Set Distribution:", count_per_category(test_dataset))

## Konvolucios háló definiálása

In [22]:
from torchvision.models import googlenet

# model = CustomCNN(num_classes)
model = googlenet(weights='IMAGENET1K_V1')  # Load pretrained weights
# for param in model.parameters():
#     param.requires_grad = False  # Fagyasztjuk az alapmodelt

# Utolsó teljesen kapcsolt réteg (fc) cseréje saját osztályszámra
model.fc = nn.Sequential(
    nn.Linear(model.fc.in_features, 512),  # Első rejtett réteg
    nn.LeakyReLU(0.01),
    nn.Dropout(0.3),
    nn.Linear(512, 256),  # Második rejtett réteg
    nn.LeakyReLU(0.01),
    nn.Dropout(0.3),
    nn.Linear(256, num_classes)  # Kimeneti réteg
)


### Logger

In [23]:
import traceback
from datetime import datetime

def log_and_upload_exception(exception, log_filename="error_log.txt", drive_folder_id=None):
    """
    Logs an exception to a file and uploads the log file to Google Drive.

    Args:
        exception (Exception): The exception object to be logged.
        log_filename (str): The name of the log file.
        drive_folder_id (str): The ID of the Google Drive folder where the log will be uploaded.
    """
    # Generate a timestamp for the log entry
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    # Create or append to the log file
    log_filepath = os.path.join("/kaggle/working", log_filename)
    with open(log_filepath, "a") as log_file:
        log_file.write(f"[{timestamp}] Exception Occurred:\n")
        log_file.write("".join(traceback.format_exception(type(exception), exception, exception.__traceback__)))
        log_file.write("\n" + "-" * 80 + "\n")

    print(f"Exception logged to {log_filepath}")

    # Upload the log file to Google Drive
    if drive_folder_id:
        try:
            file_metadata = {
                'name': log_filename,
                'parents': [drive_folder_id]  # Specify the target folder on Google Drive
            }
            media = MediaFileUpload(log_filepath, mimetype='text/plain')

            uploaded_file = drive_service.files().create(
                body=file_metadata,
                media_body=media,
                fields='id'
            ).execute()

            print(f"Log file uploaded to Google Drive (File ID: {uploaded_file.get('id')})")
        except Exception as upload_exception:
            print(f"Failed to upload the log file: {upload_exception}")


## K-fold tanítási algoritmus metrikák számításával

In [24]:
import torch
import torch.nn as nn
import torch.optim as optim
import copy
import numpy as np
import os
from collections import Counter
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import DataLoader, Subset

class CustomSubset(Subset):
    def __init__(self, dataset, indices):
        super().__init__(dataset, indices)
        self.dataset = dataset
        self.labels = [dataset[i][1] for i in indices]
    
    def __getitem__(self, idx):
        image, _ = self.dataset[self.indices[idx]]
        return image, self.labels[idx]

def stratified_kfold_split(dataset, k_folds=5):
    labels = np.array([dataset[i][1] for i in range(len(dataset))])
    while True:
        kfold = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)
        fold_splits = list(kfold.split(np.zeros(len(dataset)), labels))
        
        valid = all(len(set(labels[test_idx])) == len(set(labels)) for _, test_idx in fold_splits)
        if valid:
            return fold_splits

def train_kfold(model, dataset, num_classes=4, num_epochs=10, batch_size=32, lr=0.001, 
                k_folds=5, early_stop_threshold=3, lr_reduce_factor=0.1, lr_patience=2):

    fold_splits = stratified_kfold_split(dataset, k_folds)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    criterion = nn.CrossEntropyLoss()

    all_metrics = []
    all_histories = []
    fold_results = {}  # Store detailed results for each fold

    for fold, (train_idx, test_idx) in enumerate(fold_splits):
        print(f'Fold {fold+1}/{k_folds}')

        # Shuffle test indices
        test_idx = np.array(test_idx)
        np.random.shuffle(test_idx)
        test_idx = test_idx.tolist()
        
        val_size = len(test_idx) // 2
        val_idx = test_idx[:val_size]
        test_idx = test_idx[val_size:]

        train_subset = CustomSubset(dataset, train_idx)
        val_subset = CustomSubset(dataset, val_idx)
        test_subset = CustomSubset(dataset, test_idx)

        # Print label distribution
        print(f'Fold {fold+1} label distribution:')
        train_labels = Counter([dataset[i][1] for i in train_idx])
        val_labels = Counter([dataset[i][1] for i in val_idx])
        test_labels = Counter([dataset[i][1] for i in test_idx])
        print('Train:', train_labels)
        print('Val:', val_labels)
        print('Test:', test_labels)

        num_workers = os.cpu_count()

        train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
        val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
        test_loader = DataLoader(test_subset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

        model_fold = copy.deepcopy(model).to(device)
        optimizer = optim.Adam(model_fold.parameters(), lr=lr)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=lr_reduce_factor, patience=lr_patience)

        best_model_wts = None
        best_acc = 0.0
        early_stop_count = 0
        best_epoch = 0

        history_fold = {
            'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [],
            'val_precision': [], 'val_recall': [], 'val_f1': [], 'learning_rate': []
        }

        for epoch in range(num_epochs):
            # Training phase
            model_fold.train()
            running_loss, correct, total = 0.0, 0, 0

            for inputs_batch, labels_batch in train_loader:
                inputs_batch, labels_batch = inputs_batch.to(device), labels_batch.to(device)
                optimizer.zero_grad()
                outputs_batch = model_fold(inputs_batch)
                loss = criterion(outputs_batch, labels_batch)
                loss.backward()
                optimizer.step()

                running_loss += loss.item() * inputs_batch.size(0)
                _, predicted_batch = torch.max(outputs_batch, 1)
                correct += (predicted_batch == labels_batch).sum().item()
                total += labels_batch.size(0)

            train_loss_epoch = running_loss / total
            train_acc_epoch = correct / total

            # Validation phase
            model_fold.eval()
            val_running_loss, val_correct, val_total = 0.0, 0, 0
            y_true_val, y_pred_val = [], []

            with torch.no_grad():
                for inputs_val, labels_val in val_loader:
                    inputs_val, labels_val = inputs_val.to(device), labels_val.to(device)
                    outputs_val = model_fold(inputs_val)
                    loss_val_batch = criterion(outputs_val, labels_val)

                    val_running_loss += loss_val_batch.item() * inputs_val.size(0)
                    _, predicted_val = torch.max(outputs_val, 1)

                    val_correct += (predicted_val == labels_val).sum().item()
                    val_total += labels_val.size(0)

                    y_true_val.extend(labels_val.cpu().numpy())
                    y_pred_val.extend(predicted_val.cpu().numpy())

            val_loss_epoch = val_running_loss / val_total
            val_acc_epoch = val_correct / val_total

            # Calculate validation metrics
            target_names = [dataset.inv_label_map[i] for i in range(num_classes)]
            class_report_val = classification_report(
                y_true_val, y_pred_val, labels=list(range(num_classes)), 
                target_names=target_names, output_dict=True, zero_division=0
            )
            
            # Calculate confusion matrix for validation
            conf_matrix_val = confusion_matrix(y_true_val, y_pred_val, labels=list(range(num_classes)))

            # Extract per-class metrics
            precision_per_class_val = {f'class_{cls}': class_report_val[target_names[cls]]['precision'] for cls in range(num_classes)}
            recall_per_class_val = {f'class_{cls}': class_report_val[target_names[cls]]['recall'] for cls in range(num_classes)}
            f1_per_class_val = {f'class_{cls}': class_report_val[target_names[cls]]['f1-score'] for cls in range(num_classes)}

            current_lr = optimizer.param_groups[0]['lr']

            print(f'Epoch {epoch+1}, Train Loss: {train_loss_epoch:.4f}, Train Acc: {train_acc_epoch:.4f}, '
                  f'Val Loss: {val_loss_epoch:.4f}, Val Acc: {val_acc_epoch:.4f}, LR: {current_lr:.6f}')

            scheduler.step(val_loss_epoch)

            # Store metrics for this epoch
            history_fold['train_loss'].append(train_loss_epoch)
            history_fold['train_acc'].append(train_acc_epoch)
            history_fold['val_loss'].append(val_loss_epoch)
            history_fold['val_acc'].append(val_acc_epoch)
            history_fold['val_precision'].append(precision_per_class_val)
            history_fold['val_recall'].append(recall_per_class_val)
            history_fold['val_f1'].append(f1_per_class_val)
            history_fold['learning_rate'].append(current_lr)

            # Check if this is the best model so far
            if val_acc_epoch > best_acc:
                best_acc = val_acc_epoch
                best_model_wts = copy.deepcopy(model_fold.state_dict())
                best_epoch = epoch
                early_stop_count = 0
            else:
                early_stop_count += 1

            if early_stop_count >= early_stop_threshold:
                print(f"Early stopping at epoch {epoch+1}")
                break

        # Load best model weights
        model_fold.load_state_dict(best_model_wts)
        torch.save(best_model_wts, f'results/best_model_fold{fold}.pth')
        print(f'Best model saved for Fold {fold+1} (epoch {best_epoch+1})!')

        # Evaluate on test set
        model_fold.eval()
        test_correct, test_total = 0, 0
        y_true_test, y_pred_test = [], []
        test_probs = []  # Store probabilities for ROC curve calculation

        with torch.no_grad():
            for inputs_test, labels_test in test_loader:
                inputs_test, labels_test = inputs_test.to(device), labels_test.to(device)
                outputs_test = model_fold(inputs_test)
                probs = torch.nn.functional.softmax(outputs_test, dim=1)
                _, predicted_test = torch.max(outputs_test, 1)
                
                test_correct += (predicted_test == labels_test).sum().item()
                test_total += labels_test.size(0)

                y_true_test.extend(labels_test.cpu().numpy())
                y_pred_test.extend(predicted_test.cpu().numpy())
                test_probs.extend(probs.cpu().numpy())

        # Calculate test metrics
        test_acc = test_correct / test_total
        class_report_test = classification_report(
            y_true_test, y_pred_test, labels=list(range(num_classes)), 
            target_names=target_names, output_dict=True, zero_division=0
        )
        
        # Calculate confusion matrix for test set
        conf_matrix_test = confusion_matrix(y_true_test, y_pred_test, labels=list(range(num_classes)))
        
        # Store detailed metrics for this fold
        fold_metrics = {
            'fold': fold + 1,
            'best_epoch': best_epoch + 1,
            'train_distribution': dict(train_labels),
            'val_distribution': dict(val_labels),
            'test_distribution': dict(test_labels),
            'test_accuracy': test_acc,
            'test_report': class_report_test,
            'test_confusion_matrix': conf_matrix_test.tolist(),
            'class_names': target_names,
            # Extract per-class metrics for test set
            'test_precision': {cls: class_report_test[target_names[cls]]['precision'] for cls in range(num_classes)},
            'test_recall': {cls: class_report_test[target_names[cls]]['recall'] for cls in range(num_classes)},
            'test_f1': {cls: class_report_test[target_names[cls]]['f1-score'] for cls in range(num_classes)},
            'test_support': {cls: class_report_test[target_names[cls]]['support'] for cls in range(num_classes)},
            # Macro and weighted averages
            'test_macro_avg': class_report_test['macro avg'],
            'test_weighted_avg': class_report_test['weighted avg']
        }
        
        # Print test results
        print(f"\nFold {fold+1} Test Results:")
        print(f"Test Accuracy: {test_acc:.4f}")
        print("Classification Report:")
        print(classification_report(y_true_test, y_pred_test, labels=list(range(num_classes)), target_names=target_names))
        print("Confusion Matrix:")
        print(conf_matrix_test)
        print("\n" + "-"*50 + "\n")
        
        all_metrics.append(fold_metrics)
        all_histories.append(history_fold)
        fold_results[f'fold_{fold+1}'] = fold_metrics

    # Calculate average metrics across all folds
    avg_metrics = {
        'avg_test_accuracy': np.mean([m['test_accuracy'] for m in all_metrics]),
        'avg_test_precision': {cls: np.mean([m['test_precision'][cls] for m in all_metrics]) for cls in range(num_classes)},
        'avg_test_recall': {cls: np.mean([m['test_recall'][cls] for m in all_metrics]) for cls in range(num_classes)},
        'avg_test_f1': {cls: np.mean([m['test_f1'][cls] for m in all_metrics]) for cls in range(num_classes)},
        'avg_macro_precision': np.mean([m['test_macro_avg']['precision'] for m in all_metrics]),
        'avg_macro_recall': np.mean([m['test_macro_avg']['recall'] for m in all_metrics]),
        'avg_macro_f1': np.mean([m['test_macro_avg']['f1-score'] for m in all_metrics]),
        'avg_weighted_precision': np.mean([m['test_weighted_avg']['precision'] for m in all_metrics]),
        'avg_weighted_recall': np.mean([m['test_weighted_avg']['recall'] for m in all_metrics]),
        'avg_weighted_f1': np.mean([m['test_weighted_avg']['f1-score'] for m in all_metrics]),
    }
    
    fold_results['average'] = avg_metrics
    
    # Print average results
    print("\nAverage Results Across All Folds:")
    print(f"Average Test Accuracy: {avg_metrics['avg_test_accuracy']:.4f}")
    print(f"Average Macro F1-Score: {avg_metrics['avg_macro_f1']:.4f}")
    print(f"Average Weighted F1-Score: {avg_metrics['avg_weighted_f1']:.4f}")
    
    # Per-class average metrics
    print("\nPer-Class Average Metrics:")
    for cls in range(num_classes):
        class_name = target_names[cls]
        print(f"{class_name}: Precision={avg_metrics['avg_test_precision'][cls]:.4f}, "
              f"Recall={avg_metrics['avg_test_recall'][cls]:.4f}, "
              f"F1-Score={avg_metrics['avg_test_f1'][cls]:.4f}")

    return all_metrics, all_histories, fold_results


### K-fold alkalmazása

In [25]:
folder_id = "1LVJ2nsLuiiOB_4qJP5xbgRgYl-cs6MTU"
metrics_results = []
histories_results = []
fold_results = {}  # Új változó a harmadik visszatérési értékhez

try:
    metrics_results, histories_results, fold_results = train_kfold(model,
                    dataset,
                    num_classes=4,
                    num_epochs=25,
                    batch_size=32,
                    lr=0.001,
                    k_folds=5,
                    early_stop_threshold=5,
                    lr_reduce_factor=0.1,
                    lr_patience=3)
except Exception as e:
        log_and_upload_exception(e, drive_folder_id=folder_id)
        raise(e)

Fold 1/5
Fold 1 label distribution:
Train: Counter({1: 8154, 3: 4810, 0: 2892, 2: 1076})
Val: Counter({1: 1030, 3: 594, 0: 353, 2: 139})
Test: Counter({1: 1008, 3: 608, 0: 371, 2: 130})
Epoch 1, Train Loss: 0.5849, Train Acc: 0.7854, Val Loss: 0.4439, Val Acc: 0.8497, LR: 0.001000
Epoch 2, Train Loss: 0.4025, Train Acc: 0.8560, Val Loss: 0.4476, Val Acc: 0.8436, LR: 0.001000
Epoch 3, Train Loss: 0.3408, Train Acc: 0.8826, Val Loss: 0.5039, Val Acc: 0.8322, LR: 0.001000
Epoch 4, Train Loss: 0.2918, Train Acc: 0.8965, Val Loss: 0.4177, Val Acc: 0.8516, LR: 0.001000
Epoch 5, Train Loss: 0.2619, Train Acc: 0.9113, Val Loss: 0.3176, Val Acc: 0.8837, LR: 0.001000
Epoch 6, Train Loss: 0.2245, Train Acc: 0.9233, Val Loss: 0.3548, Val Acc: 0.8771, LR: 0.001000
Epoch 7, Train Loss: 0.2045, Train Acc: 0.9278, Val Loss: 0.3970, Val Acc: 0.8785, LR: 0.001000
Epoch 8, Train Loss: 0.1927, Train Acc: 0.9334, Val Loss: 0.4025, Val Acc: 0.8762, LR: 0.001000
Epoch 9, Train Loss: 0.1760, Train Acc: 0.9384

## Legjobb model betoltese es kiertekelese - kezdetleges, jelenleg lehet felesleges

In [26]:

# # Funkció a modell betöltésére
# def load_model(model_path):
#     model = torch.load(model_path)  # Modell betöltése (architektúra + súlyok)
#     model.eval()  # Eval módba állítás
#     print("Model loaded successfully.")
#     return model

# # Funkció a modell kiértékelésére és az eredmények mentésére Excelbe
# def evaluate_and_save_results(model, test_dataset, results_path="evaluation_results.xlsx", batch_size=32):
#     device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#     model.to(device)
    
#     # Teszt adathalmaz betöltése
#     test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
#     y_true = []
#     y_pred = []
    
#     with torch.no_grad():
#         for inputs, labels in test_loader:
#             inputs, labels = inputs.to(device), labels.to(device)
            
#             # Előrejelzés
#             outputs = model(inputs)
#             _, predicted = torch.max(outputs, 1)
            
#             y_true.extend(labels.cpu().numpy())
#             y_pred.extend(predicted.cpu().numpy())
    
#     # Metrikák kiszámítása
#     accuracy = accuracy_score(y_true, y_pred)
#     class_report = classification_report(y_true, y_pred, output_dict=True)
#     conf_matrix = confusion_matrix(y_true, y_pred)

#     # Kiértékelési eredmények mentése Excel fájlba
#     with pd.ExcelWriter(results_path, engine='openpyxl') as writer:
#         # Pontosság mentése
#         accuracy_df = pd.DataFrame({"Metric": ["Accuracy"], "Value": [accuracy]})
#         accuracy_df.to_excel(writer, sheet_name="Accuracy", index=False)

#         # Osztályonkénti metrikák mentése
#         metrics_data = []
#         for cls in class_report:
#             if cls.isdigit():  # Csak az osztályokra vonatkozó metrikák mentése
#                 metrics_data.append({
#                     "Class": cls,
#                     "Precision": class_report[cls]["precision"],
#                     "Recall": class_report[cls]["recall"],
#                     "F1-Score": class_report[cls]["f1-score"]
#                 })
#         metrics_df = pd.DataFrame(metrics_data)
#         metrics_df.to_excel(writer, sheet_name="Class Metrics", index=False)

#         # Konfúziós mátrix mentése
#         conf_matrix_df = pd.DataFrame(conf_matrix)
#         conf_matrix_df.to_excel(writer, sheet_name="Confusion Matrix", index=True)

#     print(f"Evaluation results saved to {results_path}")

# # Példa használat
# model_path = 'results/best_model.pth'  # A legjobb modell elérési útja
# test_dataset = ...  # Teszt adathalmaz (adathalmazt itt kell definiálni)

# # Modell betöltése és kiértékelése
# model = load_model(model_path)
# evaluate_and_save_results(model, test_dataset, results_path="evaluation_results.xlsx")

## Adatok mentése Google-Drive-ra

### Mentés mint excel állomány

In [27]:
def export_results_to_excel(fold_results, all_histories, filename='model_results.xlsx'):
    import pandas as pd
    
    # Create Excel writer
    with pd.ExcelWriter(filename) as writer:
        # 1. Summary sheet with average metrics
        avg_metrics = fold_results['average']
        summary_data = {
            'Metric': ['Test Accuracy', 'Macro Precision', 'Macro Recall', 'Macro F1', 
                      'Weighted Precision', 'Weighted Recall', 'Weighted F1'],
            'Value': [
                avg_metrics['avg_test_accuracy'],
                avg_metrics['avg_macro_precision'],
                avg_metrics['avg_macro_recall'],
                avg_metrics['avg_macro_f1'],
                avg_metrics['avg_weighted_precision'],
                avg_metrics['avg_weighted_recall'],
                avg_metrics['avg_weighted_f1']
            ]
        }
        
        # Add per-class average metrics
        num_classes = len(avg_metrics['avg_test_precision'])
        class_names = fold_results['fold_1']['class_names']
        
        for cls in range(num_classes):
            summary_data['Metric'].extend([
                f'{class_names[cls]} Precision',
                f'{class_names[cls]} Recall',
                f'{class_names[cls]} F1'
            ])
            summary_data['Value'].extend([
                avg_metrics['avg_test_precision'][cls],
                avg_metrics['avg_test_recall'][cls],
                avg_metrics['avg_test_f1'][cls]
            ])
        
        summary_df = pd.DataFrame(summary_data)
        summary_df.to_excel(writer, sheet_name='Summary', index=False)
        
        # 2. Per-fold metrics
        fold_metrics = []
        for fold in range(1, len(fold_results) - 1 + 1):  # Exclude 'average' key
            fold_data = fold_results[f'fold_{fold}']
            row = {
                'Fold': fold,
                'Best Epoch': fold_data['best_epoch'],
                'Test Accuracy': fold_data['test_accuracy'],
                'Macro Precision': fold_data['test_macro_avg']['precision'],
                'Macro Recall': fold_data['test_macro_avg']['recall'],
                'Macro F1': fold_data['test_macro_avg']['f1-score'],
                'Weighted Precision': fold_data['test_weighted_avg']['precision'],
                'Weighted Recall': fold_data['test_weighted_avg']['recall'],
                'Weighted F1': fold_data['test_weighted_avg']['f1-score']
            }
            
            # Add per-class metrics
            for cls in range(num_classes):
                class_name = class_names[cls]
                row[f'{class_name} Precision'] = fold_data['test_precision'][cls]
                row[f'{class_name} Recall'] = fold_data['test_recall'][cls]
                row[f'{class_name} F1'] = fold_data['test_f1'][cls]
                row[f'{class_name} Support'] = fold_data['test_support'][cls]
            
            fold_metrics.append(row)
        
        fold_metrics_df = pd.DataFrame(fold_metrics)
        fold_metrics_df.to_excel(writer, sheet_name='Fold Metrics', index=False)
        
        # 3. Confusion matrices (one sheet per fold)
        for fold in range(1, len(fold_results) - 1 + 1):
            fold_data = fold_results[f'fold_{fold}']
            conf_matrix = fold_data['test_confusion_matrix']
            conf_df = pd.DataFrame(conf_matrix, 
                                  index=[f'True {c}' for c in class_names],
                                  columns=[f'Pred {c}' for c in class_names])
            conf_df.to_excel(writer, sheet_name=f'Confusion Matrix F{fold}')
        
        # 4. Training history
        for fold, history in enumerate(all_histories, 1):
            # Basic metrics
            history_data = {
                'Epoch': list(range(1, len(history['train_loss']) + 1)),
                'Train Loss': history['train_loss'],
                'Train Acc': history['train_acc'],
                'Val Loss': history['val_loss'],
                'Val Acc': history['val_acc'],
                'Learning Rate': history['learning_rate']
            }
            
            # Add validation metrics per class for the last few epochs
            last_epochs = min(5, len(history['val_loss']))
            for cls in range(num_classes):
                class_name = class_names[cls]
                history_data[f'{class_name} Precision'] = [history['val_precision'][i][f'class_{cls}'] 
                                                          for i in range(len(history['val_precision']))]
                history_data[f'{class_name} Recall'] = [history['val_recall'][i][f'class_{cls}'] 
                                                       for i in range(len(history['val_recall']))]
                history_data[f'{class_name} F1'] = [history['val_f1'][i][f'class_{cls}'] 
                                                   for i in range(len(history['val_f1']))]
            
            history_df = pd.DataFrame(history_data)

export_results_to_excel(fold_results, histories_results, filename='results/model_results.xlsx')


In [28]:
import os
from google.oauth2 import service_account
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

# JSON kulcs fájl elérési útja
SERVICE_ACCOUNT_FILE = '/kaggle/input/googledriveuploadauth/covidcxr-0aaa95b10e20.json'

# Hitelesítés
credentials = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE,
    scopes=['https://www.googleapis.com/auth/drive']
)

# Google Drive API kliens létrehozása
drive_service = build('drive', 'v3', credentials=credentials)

# Kaggle mappa elérési útja (ahonnan fájlokat töltünk fel)
source_dir = '/kaggle/working/results'

# Szülőmappa ID a Google Drive-on (ahol az új mappa létrejön)
parent_folder_id = '1LVJ2nsLuiiOB_4qJP5xbgRgYl-cs6MTU'  # Cseréld ki a megfelelő szülőmappa ID-ra

# Google Drive célmappa neve (paraméterként megadva)
target_folder_name = 'GoogLeNET_LeakyRelu_2hiddenLayer'

# Ellenőrizd, hogy a forrásmappa létezik-e
if not os.path.exists(source_dir):
    print(f"Hiba: A '{source_dir}' mappa nem létezik.")
else:
    # Listázd az összes fájlt a mappában
    files = [os.path.join(source_dir, f) for f in os.listdir(source_dir) if os.path.isfile(os.path.join(source_dir, f))]

    # Ellenőrizd, hogy létezik-e a célmappa a megadott szülőmappában; ha nem, hozd létre
    def get_or_create_folder(folder_name, parent_id):
        # Keressük meg a mappát név és szülő ID alapján
        query = f"name='{folder_name}' and mimeType='application/vnd.google-apps.folder' and '{parent_id}' in parents"
        results = drive_service.files().list(q=query, fields="files(id, name)").execute()
        items = results.get('files', [])
        
        if items:
            # Ha létezik, térjünk vissza az ID-jával
            return items[0]['id']
        else:
            # Ha nem létezik, hozzuk létre
            file_metadata = {
                'name': folder_name,
                'mimeType': 'application/vnd.google-apps.folder',
                'parents': [parent_id]  # Szülőmappa ID-ja
            }
            folder = drive_service.files().create(body=file_metadata, fields='id').execute()
            return folder.get('id')

    # Célmappa ID lekérése vagy létrehozása a szülőmappában
    folder_id = get_or_create_folder(target_folder_name, parent_folder_id)

    # Fájlok feltöltése az újonnan létrehozott célmappába
    for file_path in files:
        file_name = os.path.basename(file_path)  # Fájl neve
        file_metadata = {
            'name': file_name,
            'parents': [folder_id]  # Célmappa azonosítója
        }
        media = MediaFileUpload(file_path, mimetype='application/octet-stream')  # MIME típus általános fájlokra

        try:
            file = drive_service.files().create(
                body=file_metadata,
                media_body=media,
                fields='id'
            ).execute()
            print(f"Feltöltve: {file_name} (File ID: {file.get('id')})")
        except Exception as e:
            print(f"Hiba történt a '{file_name}' feltöltése során: {e}")


Feltöltve: best_model_fold2.pth (File ID: 1weAwqVrwa_wKawC7x3khuLDL74p1fnOw)
Feltöltve: best_model_fold1.pth (File ID: 1J-l03dCqiqHIgZnsTave9MGiZYN71sK-)
Feltöltve: model_results.xlsx (File ID: 1x21OmnQ-5_Z2u_OMo_HDao6vprnjB7v0)
Feltöltve: best_model_fold3.pth (File ID: 1wWHNdsgvttyVO_V-0kYdsBY1WjNFUM1B)
Feltöltve: best_model_fold4.pth (File ID: 1sRlolCCHmhnC0w2GM3SUoTxJcwo9RPat)
Feltöltve: best_model_fold0.pth (File ID: 1E0o3HXoc4hEfvRaGRJ71td8KSPM9kcHO)


# Letoltes drive-rol

In [29]:
# import io
# import os
# from google.oauth2 import service_account
# from googleapiclient.discovery import build
# from googleapiclient.http import MediaIoBaseDownload

# # JSON kulcs fájl elérési útja
# SERVICE_ACCOUNT_FILE = '/kaggle/input/googledriveuploadauth/covidcxr-0aaa95b10e20.json'

# # Hitelesítés
# credentials = service_account.Credentials.from_service_account_file(
#     SERVICE_ACCOUNT_FILE,
#     scopes=['https://www.googleapis.com/auth/drive.readonly']
# )

# # Google Drive API kliens létrehozása
# drive_service = build('drive', 'v3', credentials=credentials)

# # Forrásmappa ID a Google Drive-on
# source_folder_id = '1LVJ2nsLuiiOB_4qJP5xbgRgYl-cs6MTU'  # Cseréld ki a megfelelő mappa ID-ra

# # Célmappa a letöltött fájloknak
# download_dir = '/kaggle/working/downloaded_files'
# os.makedirs(download_dir, exist_ok=True)

# # Rekurzív fájl letöltés mappából
# def download_folder_contents(folder_id, local_path):
#     # Fájlok és mappák lekérése a megadott mappából
#     query = f"'{folder_id}' in parents and trashed = false"
#     results = drive_service.files().list(
#         q=query,
#         fields="nextPageToken, files(id, name, mimeType)"
#     ).execute()
    
#     items = results.get('files', [])
    
#     if not items:
#         print(f"A mappa üres vagy nem található.")
#         return
    
#     # Fájlok és mappák feldolgozása
#     for item in items:
#         item_id = item['id']
#         item_name = item['name']
#         item_mime_type = item['mimeType']
        
#         # Elérési út létrehozása
#         item_path = os.path.join(local_path, item_name)
        
#         # Ha mappa, akkor rekurzívan letöltjük a tartalmát
#         if item_mime_type == 'application/vnd.google-apps.folder':
#             print(f"Mappa feldolgozása: {item_name}")
#             os.makedirs(item_path, exist_ok=True)
#             download_folder_contents(item_id, item_path)
#         else:
#             # Ha fájl, akkor letöltjük
#             try:
#                 request = drive_service.files().get_media(fileId=item_id)
                
#                 with open(item_path, 'wb') as f:
#                     downloader = MediaIoBaseDownload(f, request)
#                     done = False
#                     while not done:
#                         status, done = downloader.next_chunk()
#                         print(f"Letöltés: {item_name} - {int(status.progress() * 100)}%")
                
#                 print(f"Letöltve: {item_name}")
#             except Exception as e:
#                 print(f"Hiba történt a '{item_name}' letöltése során: {e}")

# # Mappa tartalmának letöltése
# print(f"A '{source_folder_id}' mappa tartalmának letöltése ide: {download_dir}")
# download_folder_contents(source_folder_id, download_dir)
# print(f"A letöltés befejeződött. A fájlok itt találhatók: {download_dir}")
